In [1]:
from llama_index.core import VectorStoreIndex
from llama_index.embeddings.langchain import LangchainEmbedding
from llama_index.readers.web import BeautifulSoupWebReader
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.core.memory import ChatMemoryBuffer

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama.llms import OllamaLLM

In [2]:
embedding_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
vector_store_name = "srh_index_store_index"
llm_name = 'llama3'
token_limit = 1000

In [3]:
with open('srh_websites.txt') as f:
    sites_to_scrape = f.read().splitlines()

In [4]:
reader = BeautifulSoupWebReader()

documents = reader.load_data(sites_to_scrape)

# Load Hugging Face embedding model
hf_embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
embedding_model = LangchainEmbedding(hf_embeddings)

# Create LlamaIndex
index = VectorStoreIndex.from_documents(documents, embed_model=embedding_model)

# persist on local disk for reuse
index.storage_context.persist(persist_dir=vector_store_name)

In [5]:
# rebuild storage context
storage_context = StorageContext.from_defaults(persist_dir=vector_store_name)

# load index
index = load_index_from_storage(storage_context, embed_model=embedding_model)

In [6]:
llm = OllamaLLM(model=llm_name, temperature=0)

In [7]:
memory = ChatMemoryBuffer.from_defaults(token_limit=token_limit)

In [8]:
chat_engine = index.as_chat_engine(
    llm=llm,
    chat_mode="context",
    memory=memory,
    system_prompt=(
        "You are a part of the SRH University. You provides academic support to university students by answering questions related to course content, providing information on academic deadlines, and offering guidance on university policies and resources. If you do not have an answer from the provided information, say so."
    ),
)

In [9]:
def generate_response(message, history):
    response = chat_engine.chat(message)
    return response.response

In [10]:
import gradio as gr

gr.ChatInterface(generate_response, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


c:\All\Misc\Data Analytics 2 - NLP\Chatbot-for-Academic-Support-to-University-Students\.env\Lib\site-packages\llama_index\llms\langchain\base.py:106: LangChainDeprecationWarning: The method `BaseLLM.predict` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  output_str = self._llm.predict(prompt, **kwargs)
